# Pointers and References

You know pointers from C. C++ adds **references** — a safer, cleaner alternative with some key differences.

By the end of this notebook you will:
- Recall pointer fundamentals from C.
- Understand what a reference is and how it differs from a pointer.
- Know when to prefer each.
- Use `const` references to pass objects efficiently.

## Pointer Review

You already know this from C — this is a quick refresh.

| Operation | Syntax | Meaning |
|---|---|---|
| Declaration | `int *p;` | `p` holds the address of an `int` |
| Address-of | `p = &x;` | store the address of `x` in `p` |
| Dereference | `*p = 42;` | write through `p` to the variable it points at |
| Pointer arithmetic | `p + 1` | address of the next `int` in memory |
| Null pointer | `p = NULL;` | `p` points to nothing (check before dereferencing!) |

In [ ]:
#include <iostream>

int value = 10;
int *ptr  = &value;

std::cout << "value    = " << value  << std::endl;
std::cout << "ptr      = " << ptr    << std::endl;   // address
std::cout << "*ptr     = " << *ptr   << std::endl;   // dereference

*ptr = 99;   // modify through pointer
std::cout << "value after *ptr = 99: " << value << std::endl;

// Pointer arithmetic on an array
int arr[3] = {10, 20, 30};
int *ap    = arr;
for (int i = 0; i < 3; i++) {
    std::cout << "arr[" << i << "] via pointer arithmetic = " << *(ap + i) << std::endl;
}

## References: A New Concept

A **reference** is an alias — another name for an already-existing variable.

```cpp
int  x   = 42;
int &ref = x;   // ref IS x — not a pointer to x
```

Key rules:
- A reference **must be initialised** at the point of declaration.
- A reference **cannot be reseated** — once it refers to a variable, it always refers to that same variable.
- You use `.` for member access (not `->`).
- No `*` needed to read/write through it — it behaves exactly like the original variable.

In [ ]:
#include <iostream>

int x   = 42;
int &ref = x;   // ref is an alias for x

std::cout << "x   = " << x   << std::endl;
std::cout << "ref = " << ref << std::endl;

ref = 100;   // modifies x through the alias
std::cout << "After ref = 100:" << std::endl;
std::cout << "x   = " << x   << std::endl;
std::cout << "ref = " << ref << std::endl;

// Both have the same address
std::cout << "&x   = " << &x   << std::endl;
std::cout << "&ref = " << &ref << std::endl;
std::cout << "Same address? " << (&x == &ref ? "YES" : "NO") << std::endl;

## References vs Pointers — Key Differences

| Feature | Reference | Pointer |
|---|---|---|
| Initialisation | Must be initialised at declaration | Can be declared uninitialised (dangerous) |
| Null value | Cannot be null (always valid) | Can be `NULL` / `0` |
| Reseating | Cannot point to a different variable | Can point to different variables over time |
| Arithmetic | No reference arithmetic | Pointer arithmetic is valid |
| Member access | `ref.member` | `ptr->member` |
| Dereference syntax | `ref` (use directly) | `*ptr` |
| Dangling risk | Lower (if used correctly) | Higher (null, dangling, double-free) |

## Pass by Value, Pointer, Reference

A classic example — swapping two integers — shows the difference clearly.

In [ ]:
#include <iostream>

// Pass by value -- does NOT swap the originals
void swapByValue(int a, int b) {
    int tmp = a;
    a = b;
    b = tmp;
    // a and b are copies; originals untouched
}

// Pass by pointer -- C style
void swapByPointer(int *a, int *b) {
    int tmp = *a;
    *a = *b;
    *b = tmp;
}

// Pass by reference -- C++ style (cleaner)
void swapByReference(int &a, int &b) {
    int tmp = a;
    a = b;
    b = tmp;
}

int p = 1, q = 2;

swapByValue(p, q);
std::cout << "After swapByValue:     p=" << p << " q=" << q << std::endl;

swapByPointer(&p, &q);
std::cout << "After swapByPointer:   p=" << p << " q=" << q << std::endl;

swapByReference(p, q);
std::cout << "After swapByReference: p=" << p << " q=" << q << std::endl;

**Exercise 1:** Write a function `void addTen(int &n)` that adds 10 to its argument in-place using a reference parameter. Call it with a variable and verify the original was modified.

In [ ]:
// Your code here

## const References

```cpp
const int &ref = x;   // read-only alias
```

A `const` reference lets you read the value but not modify it through the alias.

### The standard idiom for passing objects

```cpp
void printName(const std::string &name);   // preferred
void printName(std::string name);          // copies the whole string -- wasteful
void printName(std::string *name);         // C style -- needs NULL check, ugly call
```

Passing by `const &`:
- Avoids copying the object (fast for large objects).
- Prevents accidental modification.
- Makes the call site clean: `printName(myStr)` — no `&` needed.

In [ ]:
#include <iostream>
#include <string>

void printName(const std::string &name) {
    // name = "cannot do this";   // compile error: const reference
    std::cout << "Name: " << name << std::endl;
}

int computeLength(const std::string &s) {
    return (int)s.size();
}

std::string myName = "42 student";
printName(myName);
std::cout << "Length: " << computeLength(myName) << std::endl;

// const ref can bind to a temporary (literal)
printName("temporary string");

## References to Class Objects

References work with class objects just as with primitive types. Pass by reference to modify the original; pass by `const` reference to read it safely.

In [ ]:
#include <iostream>

class Box {
public:
    double width;
    double height;
    double depth;

    Box(double w, double h, double d) : width(w), height(h), depth(d) {}

    double volume() const { return width * height * depth; }
};

// Modifies the original through a reference
void doubleBox(Box &b) {
    b.width  *= 2;
    b.height *= 2;
    b.depth  *= 2;
}

// Read-only access -- uses const reference
void describeBox(const Box &b) {
    std::cout << "Box " << b.width << "x" << b.height << "x" << b.depth
              << " volume=" << b.volume() << std::endl;
}

Box myBox(2.0, 3.0, 4.0);
describeBox(myBox);
doubleBox(myBox);
describeBox(myBox);

**Exercise 2:** Write a function `void scaleBox(Box &b, double factor)` that multiplies all three dimensions of `b` by `factor` in-place. Write a separate `void printBox(const Box &b)` function that prints the box dimensions. Test both.

In [ ]:
// Your code here

## Returning References

You can return a reference from a function, which allows **chaining** and **in-place access** to members.

### When it is correct
- Returning a reference to a member of the object (`return _data;`).
- Returning `*this` from `operator=` (as in OCF).

### DANGER: Never return a reference to a local variable

```cpp
int &badFunction() {
    int localVar = 42;
    return localVar;   // UNDEFINED BEHAVIOUR -- localVar is destroyed on return
}
```

The local variable is destroyed when the function returns; the caller holds a dangling reference.

In [ ]:
#include <iostream>

class NumberHolder {
public:
    NumberHolder(int v) : _value(v) {}

    // Correct: returning reference to member
    int &value()             { return _value; }
    const int &value() const { return _value; }

private:
    int _value;
};

NumberHolder holder(10);
std::cout << "value = " << holder.value() << std::endl;

holder.value() = 99;   // assign through the returned reference
std::cout << "value after holder.value() = 99: " << holder.value() << std::endl;

## Pointers to Objects

When an object is heap-allocated with `new`, you access it through a pointer and use the arrow operator `->`.

In [ ]:
#include <iostream>

class Point {
public:
    int x, y;

    Point(int x, int y) : x(x), y(y) {}

    void print() const {
        std::cout << "Point(" << x << ", " << y << ")" << std::endl;
    }
};

Point *ptr = new Point(3, 7);

ptr->print();                          // arrow operator for member function
std::cout << "x = " << ptr->x << std::endl;   // arrow operator for member variable

// Equivalent using dereference:
(*ptr).print();   // same as ptr->print()

delete ptr;       // always delete what you new
ptr = NULL;

## Reference vs Pointer in Practice

**Use a reference when:**
- The argument is always valid (never null).
- You do not need to reseat it.
- You want clean call syntax.

**Use a pointer when:**
- The value can legitimately be null (`NULL` / not provided).
- You need to point to different objects over time.
- You are working with arrays or doing pointer arithmetic.
- You are managing heap memory with `new` / `delete`.

In [ ]:
#include <iostream>
#include <string>

class Student {
public:
    std::string name;
    int         grade;

    Student(const std::string &n, int g) : name(n), grade(g) {}
};

// Reference: always valid, read-only
void printStudent(const Student &s) {
    std::cout << s.name << " -> grade " << s.grade << std::endl;
}

// Reference: modify in-place
void promote(Student &s) {
    s.grade += 1;
}

// Pointer: could be NULL (optional)
void printOptionalStudent(const Student *s) {
    if (s == NULL)
        std::cout << "(no student)" << std::endl;
    else
        std::cout << s->name << " -> grade " << s->grade << std::endl;
}

Student alice("Alice", 3);
printStudent(alice);
promote(alice);
printStudent(alice);
printOptionalStudent(NULL);
printOptionalStudent(&alice);

## Final Exercise

**Exercise 3:** Write a function:

```cpp
bool findAndReplace(std::string &text,
                    const std::string &target,
                    const std::string &replacement);
```

The function should:
- Search `text` for the first occurrence of `target`.
- If found, replace it with `replacement` **in-place** (modifying `text` through the reference).
- Return `true` if a replacement was made, `false` otherwise.

Hint: `std::string::find` returns `std::string::npos` if not found. Use `std::string::replace` to do the substitution.

Test with at least two cases: one where `target` exists and one where it does not.

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

### `nullptr` instead of `NULL`

C++11 introduces `nullptr` — a null pointer constant with type `std::nullptr_t`. It avoids ambiguity that `NULL` (which is just `0`) can cause with overloaded functions.

```cpp
int *p = nullptr;   // preferred over NULL in C++11
```

### Rvalue References `&&`

An rvalue reference binds to a **temporary** (an rvalue — something without a persistent address). This enables move semantics: instead of copying data, you "steal" it from a temporary that is about to be destroyed.

```cpp
void process(std::string &&s);   // takes ownership of a temporary string
```

### `std::move`

`std::move(x)` casts `x` to an rvalue reference, allowing its resources to be moved rather than copied.

In [ ]:
#include <iostream>
#include <string>
#include <utility>   // std::move

void processString(std::string &&s) {
    std::cout << "Received (moved): " << s << std::endl;
    // s is now ours to use or destroy
}

std::string greeting = "Hello from 42";
std::cout << "Before move: greeting = \"" << greeting << "\"" << std::endl;

processString(std::move(greeting));   // transfer ownership

// greeting is now in a valid but unspecified state (likely empty)
std::cout << "After  move: greeting = \"" << greeting << "\"" << std::endl;

// nullptr example
int *safePtr = nullptr;
std::cout << "safePtr is nullptr? " << (safePtr == nullptr ? "yes" : "no") << std::endl;